In [9]:
import pyshacl
from rdflib import OWL, RDF, RDFS, SH, Graph, Namespace
import rdflib


# running in si-builder venv

In [2]:
S223 = Namespace("http://data.ashrae.org/standard223#")

In [3]:
deactivate_info = """

prefix ex: <http://example.org#> 
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT 
{ ?rule sh:deactivated true }
WHERE
{

  { ?rule sh:severity sh:Info .}
  UNION
  {?rule sh:severity sh:Warning.}
}
"""

In [ ]:
g = Graph()
g.parse('test_model.ttl', format = 'ttl')

g.parse('models/core.ttl', format = 'ttl')
g.parse('models/equipment.ttl', format = 'ttl')
# g.parse('vocab/electricity.ttl', format = 'ttl')
g.parse('vocab/enumeration.ttl', format = 'ttl')
g.parse('inference/model-rules.shapes.ttl', format = 'ttl')
g.parse('inference/owl-subset.shapes.ttl', format = 'ttl')
g.parse('inference/data-rules.shapes.ttl', format = 'ttl')
g.parse('validation/model.shapes.ttl', format = 'ttl')
g.parse('validation/data.shapes.ttl', format = 'ttl')
g.parse('validation/schema.shapes.ttl', format = 'ttl')

g = g.query(deactivate_info).graph + g

# g.skolemize()

og = Graph()
pyshacl.rdfutil.clone.clone_graph(g, og)
sg = Graph()
pyshacl.rdfutil.clone.clone_graph(g, sg)

valid, report_graph, report_text = pyshacl.validate(
    data_graph=g,
    shacl_graph=sg,
    advanced=True,
    js=True,
    inplace=True,
    iterate_rules=True
)

valid, report_graph, report_text = pyshacl.validate(
    data_graph=g,
    shacl_graph=sg,
    advanced=True,
    js=True,
    inplace=True,
    iterate_rules=True
)

# 'ex:Generator' in report_text
# print(report_text)

# find the prefix definitions so the select can find them
namespace_map = {}
for prefix, uriref in report_graph.namespaces():
    namespace_map[prefix] = Namespace(uriref)
if "sh" not in namespace_map:
    namespace_map["sh"] = SH

# find the validation results
# Currently filtering out info in qs
qs = """
    SELECT ?resultSeverity ?sourceShape ?resultMessage ?focusNode ?value
    WHERE {
        ?report rdf:type sh:ValidationReport .
        ?report sh:result ?result .
        ?result sh:focusNode ?focusNode .
        OPTIONAL { ?result sh:resultMessage ?resultMessage } .
        ?result sh:resultSeverity ?resultSeverity .
        ?result sh:sourceShape ?sourceShape .
        OPTIONAL { ?result sh:value ?value } .
        # FILTER(?resultSeverity = s223:g36).
        }
    """

# pretty colors
color_map = {SH.Violation: 33, SH.Info: 34, SH.Warning: 35}

# run the query, sort the results
results = sorted(report_graph.query(qs, initNs=namespace_map))

prev = None
for resultSeverity, sourceShape, resultMessage, focusNode, value in results:
    if sourceShape != prev:
        color = color_map[resultSeverity]
        if resultMessage:
            print(f"\x1b[{color}m{resultMessage}\x1b[0m")
        # else:
        print(f"\x1b[{color}m{sourceShape}\x1b[0m")
        prev = sourceShape
    print(f"    {focusNode}{' ' + value if value else ''}")


In [18]:
for triple in g.triples((None, None, rdflib.term.BNode("n50b81c7a582a4402b952be8480f7dcf4b1"))):
    print(triple)

In [ ]:
g.serialize('temp.ttl', format = 'ttl')

<Graph identifier=Nf93dc4e3762f4aca929834b2fb4c115f (<class 'rdflib.graph.Graph'>)>

In [16]:
added_triples = g - og
added_triples.print()

In [17]:
print(report_text)

Validation Report
Conforms: True



In [18]:
# error locator

In [47]:
temp_file = 'temp.ttl'
def read_file_exclude_after_last_period(file_path):
    with open(file_path, 'r') as file:
        content = file.read()

    last_period_index = content.rfind('.')
    second_last_period_index = content.rfind('.', 0, last_period_index)

    if last_period_index != -1 and second_last_period_index != -1:
        modified_content = content[:second_last_period_index+1]  # Include the second last period
        problem = content[second_last_period_index:last_period_index]
    else:
        modified_content = content

    with open(file_path, 'w') as file:
        file.write(modified_content)
def copy_file(source_file, destination_file):
    with open(source_file, 'rb') as src, open(destination_file, 'wb') as dst:
        dst.write(src.read())

In [48]:
copy_file('extensions/G36_SP223-v1.0.ttl', 'temp.ttl')
for i in range(0,1000):
    problem = read_file_exclude_after_last_period(temp_file)
    try:
        g = Graph()
        g.parse(temp_file, format = 'ttl')
        g = g.query(deactivate_info).graph + g

        valid, report_graph, report_text = pyshacl.validate(
            data_graph=g,
            shacl_graph=g,
            advanced=True,
            js=True,
            allow_warnings=False,
            inplace=True,
            iterate_rules=True,
        )
    except Exception as e:
        print(e)
        continue
    print('no exception')
    break
    

FileNotFoundError: [Errno 2] No such file or directory: 'extensions/G36_SP223-v1.0.ttl'